# Levels where NCHW / NHWC exists

Think in layers of abstraction.

$$
\text{Model Graph} \\
\downarrow \\
\text{Tensor IR / Graph IR} \\
\downarrow \\
\text{Kernel selection + Lowering} \\
\downarrow \\
\text{Memory layout + Hardware}
$$

#### NCHW / NHWC appear at $\text{multiple layers}$, not just one

## High-Level (model/graph level)

Example:

In [ ]:
x = Conv2D(data_format="NHWC")(x)

Here:
- You are declaring semantic layout
- No guarantee about physical memory yet
- This is logical layout

Used by:
- TensorFlow graph
- ONNX
- XLA / MLIR
- CoreML spec

At this level:
- Layout is metadata
- Optimizers may rewrite it

#### $\rightarrow$ Yes, NHWC/NCHW exist at high level

# Tensor IR / Compiler Level (where it becomes serious)

Framework compilers operate here:
- XLA
- TVM
- TorchInductor
- MLIR
- Glow

At this level:
- Layout affects:
    - Operator fusion
    - Tiling strategy
    - Vectorization
- Layout transforms are explicit IR ops

Example (conceptual):

conv_nchw → transpose → relu_nhwc → fuse?

This is where:
- Layout decisions become optimization decisions
- Transposes are inserted or eliminated

#### $\rightarrow$ Still not raw pointers yet.

# Kenrel Level (where it becomes real)

This is where layout is baked into code

#### Example CUDA kernel assumption

idx = ((n * C + c) * H + h) * W + w;

or

idx = ((n * H + h) * W + w) * C + c;

At this level:
- Layout is hard-coded
- Kernel only works for one layout
- Changing layout = different kernel

Libraries:
- cuDNN
- oneDNN
- MPS
- Metal Performance Shaders
- CoreML runtime

#### $\rightarrow$ Here layout is physical and non-negotiable

# Memory Level (what the hardware sees)

Ultimately:
- RAM / VRAM is just a flat byte array
- NCHW / NHWC is address arithmetic

No “format” exists in hardware.

Only:
$$
pointer + stride
$$

Layout = strides:
- NCHW: large stride on C
- 